![Astrofisica Computacional](../../../new_logo.png)

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Astronomical Institute of the Czech Academy of Sciences\
   Department of Solar Physics\
   Ondřejov, Czec Republic

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & rozo@asu.cas.cz)

---

Advanced NumPy Exercises + Functions/Classes + Files

**Dataset:** Adapted from the Helsinki 2017 Temperatures dataset (https://raw.githubusercontent.com/csmastersUH/data_analysis_with_python_2020/master/kumpula-weather-2017.csv)

Includes:
- NumPy: slicing, masks, polyfit, FFT.

- Functions/Classes: inheritance, decorators, vectorized methods.

- Files: loadtxt, processed savetxt.

In [37]:
import numpy as np
import pandas as pd

## Exercise 1: Data Loading + NumPy Preprocessing

Load temps from the file "kumpula-weather-2017.csv". Write a function to convert mm-dd to days since start (days = monthday_to_days).

- Filter NaNs or infs/masks.

- Calculate anomalies (temperature - 30-day moving average).

- Detect outliers (IQR method, Q1-1.5*IQR).

**Expected output:** `temps_clean` (N x 2), `dias` (N,).

In [38]:
#TODO
#importar datos
url = "https://raw.githubusercontent.com/csmastersUH/data_analysis_with_python_2020/master/kumpula-weather-2017.csv"
dr = pd.read_csv(url)
data = dr.select_dtypes(include=[float, int]).to_numpy()
labels=['Año', 'Dia_del_año',"Precipitation (mm)","Snow depth (cm)",'Temperatura(c)']
#data.shape
#mask
mask=np.all(np.isfinite(data),axis=1) # el np.isfinite devuelve un array booleano con True donde el valor es finito y False donde no lo es, y el np.all devuelve True si todos los elementos del array booleano son True, 
                                #lo que significa que todos los valores en el array original son finitos.

data=data[mask] # se aplica la mascara al array original para obtener solo las filas que contienen valores finitos, se borra el resto de filas que contienen valores no finitos (NANs, infs).
                  
#mm-dd to days
def meses_a_dias(m, d):
    dias_acumulados = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
    return dias_acumulados[int(m) - 1] + d

V_meses_a_dias = np.vectorize(meses_a_dias)
data[:, 1] = V_meses_a_dias(data[:, 1], data[:, 2])
data[:, 2] = 0

data=np.delete(data, 2, axis=1)
#anomalias
def anomalias(data,window):  #distancia de la media movil
    w=np.ones(window)/window
    media_movil=np.convolve(data,w,mode='same') # si se escoje 'same' se generan datos falsos en las esquinas, se soluciona con valid pero se pierde informacion.
    anomalia=data-media_movil
    return anomalia
# IQR
temp=data[:, 4] # Temperatura
anomal=anomalias(temp,30)
q1 = np.percentile(anomal, 25)
q3 = np.percentile(anomal, 75)
iqr = q3 - q1
limite_bajo = q1 - 1.5 * iqr
limite_alto = q3 + 1.5 * iqr
print(np.size(temp))
mask_normal = (anomal >= limite_bajo) & (anomal <= limite_alto) #limpio anomalias
temp=temp[mask_normal]
print(np.size(temp))
anomal=anomal[mask_normal]
data = data[mask_normal]

temps_clean = np.column_stack((temp, anomal))
days=data[:, 1]

#final
data=pd.DataFrame(data,columns=labels)
print('datos Limpios')
print(data.head())
print('temperatura')
print(np.round(temps_clean[:10],2))
print('dias')
print(days[:10])

358
348
datos Limpios
      Año  Dia_del_año  Precipitation (mm)  Snow depth (cm)  Temperatura(c)
0  2017.0          1.0                -1.0             -1.0             0.6
1  2017.0          2.0                 4.4             -1.0            -3.9
2  2017.0          3.0                 6.6              7.0            -6.5
3  2017.0          7.0                 5.3             10.0            -3.8
4  2017.0          8.0                -1.0             12.0            -0.5
temperatura
[[ 0.6   2.79]
 [-3.9  -1.57]
 [-6.5  -4.06]
 [-3.8  -1.37]
 [-0.5   1.9 ]
 [ 0.5   2.9 ]
 [ 1.7   4.17]
 [-1.6   1.  ]
 [-2.8  -0.27]
 [ 1.1   3.58]]
dias
[ 1.  2.  3.  7.  8.  9. 10. 11. 12. 13.]


## Exercise 2: Advanced Class with Inheritance/Decorators 

Create a **base class TimeSeriesAnalyzer**:
- `smooth(self, window=7)`: moving average.

- Decorator `@vectorize` to apply functions to columns.

**Child class WeatherAnalyzer** inherits and adds:
- `seasonal_decompose(self)`: trend (polyfit deg=2), seasonal (FFT top 4 freq), residual.

- `forecast(self, days_ahead=30)`: simple ARIMA-like (last 30 days, polyfit deg=3) + noise.

- **Use:** `analyzer = WeatherAnalyzer(days, temps_clean[:,1])`

In [35]:
import numpy as np


# =========================================================
# DECORATOR: vectorize
# =========================================================
# Esta función es un decorator que permite aplicar un método
# que trabaja sobre UNA serie temporal a TODAS las columnas
# de la matriz de datos automáticamente.
#
# Esto evita for's.
# =========================================================

def vectorize(method):

    def wrapper(self, *args, **kwargs):

        # self.data tiene forma (N, M)
        # N = Total de dias
        # M = temp, profundiad de la nieve etc etc

        # .T transpone la matriz -> (M, N)
        # Esto permite iterar sobre columnas
        results = np.array([
            method(self, col, *args, **kwargs)   # se aplica la función a cada columna
            for col in self.data.T
        ]).T

        # El resultado de la lista queda como (M, N)
        # entonces se transpone nuevamente para regresar
        # al formato original (N, M)

        return results

    return wrapper



# =========================================================
# CLASE BASE: TimeSeriesAnalyzer
# =========================================================
# Esta clase contiene herramientas básicas para analizar
# series temporales:
#
# - suavizado
# - estadísticas básicas
# =========================================================

class TimeSeriesAnalyzer:


    # -----------------------------------------------------
    # CONSTRUCTOR
    # -----------------------------------------------------
    # Se ejecuta cuando creamos el objeto
    #
    # analyzer = WeatherAnalyzer(days, temps)
    #
    # Guarda los datos y verifica consistencia.
    # -----------------------------------------------------
    def __init__(self, t, data):

        # se convierten a arrays de numpy
        self.t = np.asarray(t)
        self.data = np.asarray(data)

        # chequeo de consistencia
        if self.t.shape[0] != self.data.shape[0]:
            raise ValueError("t y data deben tener misma longitud")


    # -----------------------------------------------------
    # SMOOTH
    # -----------------------------------------------------
    # Aplica un promedio móvil para suavizar la señal.
    #
    # Matemáticamente:
    #
    # y_i = (1/w) Σ x_{i-k}
    #
    # donde w es el tamaño de la ventana.
    #
    # El decorator @vectorize hace que esto se aplique
    # automáticamente a todas las columnas.
    # -----------------------------------------------------
    @vectorize
    def smooth(self, col, window=7, mode="same"):

        # kernel del promedio móvil
        kernel = np.ones(window) / window

        # convolución -> suaviza la señal
        return np.convolve(col, kernel, mode=mode)


    # -----------------------------------------------------
    # STATS
    # -----------------------------------------------------
    # Calcula estadísticas básicas de cada serie:
    #
    # - media
    # - desviación estándar
    # - mínimo
    # - máximo
    #
    # axis=0 significa que se calculan a lo largo
    # del tiempo para cada columna.
    # -----------------------------------------------------
    def stats(self):

        return {

            "mean": np.mean(self.data, axis=0),

            "std": np.std(self.data, axis=0),

            "min": np.min(self.data, axis=0),

            "max": np.max(self.data, axis=0)

        }



# =========================================================
# CLASE DERIVADA: WeatherAnalyzer
# =========================================================
# Hereda las funciones de TimeSeriesAnalyzer
# y agrega métodos específicos para análisis climático.
# =========================================================

class WeatherAnalyzer(TimeSeriesAnalyzer):


    # -----------------------------------------------------
    # SEASONAL_DECOMPOSE
    # -----------------------------------------------------
    # Descompone la serie en tres partes:
    #
    # data = trend + seasonal + residual
    #
    # trend -> tendencia lenta
    # seasonal -> variaciones periódicas
    # residual -> ruido
    # -----------------------------------------------------
    def seasonal_decompose(self, degree_trend=2):


        # -------------------------------------------------
        # CAMBIO RESPECTO AL ORIGINAL
        # -------------------------------------------------
        # El código original tenía errores al manejar
        # las dimensiones de FFT.
        #
        # También intentaba aplicar polyfit a matrices
        # completas, lo cual genera errores.
        #
        # Por eso aquí se hace columna por columna.
        # -------------------------------------------------


        trend = np.zeros_like(self.data)


        # ===============================================
        # calcular tendencia usando ajuste polinomial
        # ===============================================
        for i in range(self.data.shape[1]):

            # ajuste polinomial
            p = np.polyfit(self.t, self.data[:, i], degree_trend)

            # evaluación del polinomio
            trend[:, i] = np.polyval(p, self.t)



        # ===============================================
        # quitar tendencia
        # ===============================================
        detrended = self.data - trend



        # ===============================================
        # extraer componente periódica usando FFT
        # ===============================================
        fft = np.fft.fft(detrended, axis=0)

        seasonal = np.real(np.fft.ifft(fft, axis=0))



        # ===============================================
        # calcular residual
        # ===============================================
        residual = self.data - trend - seasonal



        return {

            "trend": trend,

            "seasonal": seasonal,

            "residual": residual

        }



    # -----------------------------------------------------
    # FORECAST
    # -----------------------------------------------------
    # Predice valores futuros usando un ajuste polinomial
    # sobre los datos más recientes.
    #
    # También agrega ruido gaussiano para simular
    # variabilidad natural.
    # -----------------------------------------------------
    def forecast(self, days_ahead=30, degree=3, noise_std=1.0):


        # número de puntos usados para ajustar
        n_last = min(degree*10, len(self.t)//2)



        t_last = self.t[-n_last:]

        data_last = self.data[-n_last:]



        # tiempos futuros
        t_future = np.arange(self.t[-1] + 1,
                             self.t[-1] + days_ahead + 1)



        # -------------------------------------------------
        # CAMBIO IMPORTANTE
        # -------------------------------------------------
        # El código creaba:
        #
        # forecast = np.zeros(days_ahead)
        #
        # lo cual solo sirve para UNA serie.
        #
        # Aquí lo cambiamos para múltiples columnas.
        # -------------------------------------------------
        forecast = np.zeros((days_ahead, self.data.shape[1]))



        # ===============================================
        # ajuste polinomial para cada serie
        # ===============================================
        for i in range(self.data.shape[1]):


            # CAMBIO:
            # polyfit debe recibir vectores,
            # no matrices completas
            p = np.polyfit(t_last, data_last[:, i], degree)


            forecast[:, i] = (

                np.polyval(p, t_future)

                + np.random.normal(0, noise_std, days_ahead)

            )



        return t_future, forecast



# =========================================================
# USO DEL CÓDIGO
# =========================================================

# creación del objeto
analyzer = WeatherAnalyzer(days, temps_clean)


# suavizado
smoothed = analyzer.smooth(window=15)


# descomposición
decomp = analyzer.seasonal_decompose()


# predicción
t_fc, fc = analyzer.forecast(30)


# estadísticas


print(analyzer.stats())

{'mean': array([6.87155172, 0.19254789]), 'std': array([6.76874332, 2.30045475]), 'min': array([-7.3       , -6.39333333]), 'max': array([19.6 ,  6.59])}


## Exercise 3: Robust I/O + Processed 

- Save `temps_clean` + `anomalies` to 'processed_weather.npz' (np.savez).

- Write subset (first 100 days, columns: days, temp_smooth, anomaly) to 'subset.csv' (savetxt, fmt='%.2f', header).

- Function `load_and_validate(filename)`: loads npz/csv, checks shape/inf/NaNs, returns a dictionary.

In [36]:

# guardar arrays en un archivo comprimido .npz

np.savez("processed_weather.npz", temps=temps_clean,anomal=anomal)

# crea el subconjunto
subset = np.column_stack([days[:100], smoothed[:100,0], anomal[:100]])
np.savetxt('subset.csv', subset, delimiter=',', header='day,temp_smooth,anomaly', fmt='%.3f')

def load_and_validate(filename):

    data_dict = {}

    # -------------------------------------------------
    # cargar archivos NPZ
    # -------------------------------------------------
    if filename.endswith(".npz"):

        data = np.load(filename)

        for key in data.files:

            arr = data[key]

            # validar NaN o infinitos
            if np.isnan(arr).any() or np.isinf(arr).any():
                raise ValueError(f"Invalid values in {key}")

            data_dict[key] = arr


    # -------------------------------------------------
    # cargar archivos CSV
    # -------------------------------------------------
    elif filename.endswith(".csv"):

        arr = np.loadtxt(filename, delimiter=",", skiprows=1)

        if np.isnan(arr).any() or np.isinf(arr).any():
            raise ValueError("CSV contains invalid values")

        data_dict["data"] = arr


    else:

        raise ValueError("Unsupported file type")


    return data_dict


print(load_and_validate("subset.csv"))

{'data': array([[ 1.000e+00, -9.000e-01,  2.787e+00],
       [ 2.000e+00, -1.087e+00, -1.573e+00],
       [ 3.000e+00, -1.013e+00, -4.057e+00],
       [ 7.000e+00, -9.600e-01, -1.367e+00],
       [ 8.000e+00, -1.147e+00,  1.900e+00],
       [ 9.000e+00, -1.427e+00,  2.897e+00],
       [ 1.000e+01, -1.660e+00,  4.170e+00],
       [ 1.100e+01, -1.587e+00,  9.970e-01],
       [ 1.200e+01, -1.520e+00, -2.670e-01],
       [ 1.300e+01, -1.300e+00,  3.580e+00],
       [ 1.400e+01, -9.870e-01,  3.253e+00],
       [ 1.500e+01, -6.670e-01, -3.670e-01],
       [ 1.600e+01, -6.270e-01, -1.800e+00],
       [ 1.700e+01, -8.070e-01, -1.087e+00],
       [ 1.800e+01, -1.173e+00,  3.403e+00],
       [ 1.900e+01, -9.400e-01,  3.713e+00],
       [ 2.000e+01, -6.470e-01,  1.093e+00],
       [ 2.100e+01, -6.670e-01, -7.130e-01],
       [ 2.200e+01, -6.800e-01,  1.577e+00],
       [ 2.300e+01, -4.270e-01,  7.930e-01],
       [ 2.400e+01, -1.330e-01, -1.120e+00],
       [ 2.500e+01,  6.000e-02, -2.410e+00],
 

# Some Questions:

- For outliers (IQR method): what fraction of data is removed? Is this conservative/aggressive? Propose alternative (e.g., sigma=3).
- Trace @vectorize execution: for data.shape=(365,2), how many calls to smooth(col)? Why results.T? What other vectorizing method you propose?
- In seasonal_decompose(): why subtract trend before FFT? What do top-4 frequencies represent (daily/weekly/monthly/yearly)?
- Run analyzer.stats() on raw vs smoothed: % std reduction per column? Why does polyfit deg=2 capture trend well?

# 1
la fraccion seria datos_remove/datosTotal para este ejemplo fueron  10/358 lo que es un 2.8% de datos removidos, para mi es conservador. El metodo 3 sigma seria lo ideal a mi forma de ver ya que elimina solo el 0.27% de los datos.
# 2
la funcion smooth tiene 2 llamados para data.shape= (365.2), y el resultado transpuesto se usa por que la data que recibe wrapped entra traspuesta para tener cada variable en una fila y a la salida la vuelve a trasponer (D^T)^T=D para recuperar la data original, (reemplaza a un ciclo for, ejemplo for days in d: extraer temp).
metodo de vectorizacion @ np.vectorize o algo con la misma logica de la funcion del ejercicio np.apply_along_axis.
# 3
Se resta trend debio a que se esta realizando una trasformada de fourier y si los datos tienen una tendencia fuerte o marcada (trend calcula la tendencia total, hace una curva suave que "sigue" los datos), al realizar fourier apareceria ese "ruido" como oscilaciones inpidiendo la observacion de la frecuencia real. para este caso las 4 señales principales de la FFT son las frecuencias en dia-noche, semanas, meses y años.


In [40]:
#4 
raw_data= analyzer.stats()
analyzer_smooth = WeatherAnalyzer(days, smoothed)
stats_smooth = analyzer_smooth.stats()
std_raw = raw_data["std"]
std_smooth = stats_smooth["std"]
# calcular reducción porcentual
reduccion = (std_raw - std_smooth) / std_raw * 100

# mostrar resultados
print("Desviación estándar original:", std_raw)
print("Desviación estándar suavizada:", std_smooth)
print("Reducción porcentual (%):", reduccion) 

Desviación estándar original: [6.76874332 2.30045475]
Desviación estándar suavizada: [6.39750049 0.74486537]
Reducción porcentual (%): [ 5.484664   67.62095092]


# 4
Un polinomio de grado 2 captura bien la tendencia porque permite describir una variación suave con curvatura en el tiempo.Esto es suficiente para modelar cambios lentos en la serie temporal sin sobreajustar el ruido de los datos. si se usara poli fit 1 no seguiria bien los datos y para poli fit >2 empieza a seguir el ruido 